In [ ]:
# Imports for the modeling notebook.
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable, bioDraws, MonteCarlo, exp, log, Elem, bioNormalCdf
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
import pandas as pd
import biogeme.database as db
import numpy as np
import biogeme.distributions as dist
import pickle
from urllib.request import urlopen
import os


In [ ]:
# Load the cleaned dataset produced by `loading_and_cleaning_data.ipynb`.
# low_memory=False suppresses the mixed-type DtypeWarning on the wide CSV.
df = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)


## Sanity check and preparing the databases

In [ ]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

continuous_vars = [
    'age',
    'Number of passengers',
    'number of involved vehicles',
    'vma',
    'age_2',
    'age_opposite_mean'
]

df_centered = df.copy()

df_centered[continuous_vars] = (
    df_centered[continuous_vars]
    - df_centered[continuous_vars].mean()
)
df_centered = df.copy()


df_non_dummies = df_centered[['age','severity','Number of passengers','number of involved vehicles','vma','age_2','catu', 'Num_Acc','age_opposite_mean']]




In [ ]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(999)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [ ]:
## First model
df_carcrashes=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_carcrashes=df_carcrashes.loc[df_carcrashes['catu'].isin([1,2])]
database_carcrashes = db.Database('database_carcrashes', df_carcrashes)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
df_mmv=df_mmv.loc[df_mmv['severity']!=3] ## Only one value
df_mmv = df_mmv.loc[df_mmv['age_opposite_mean'] != 999].copy()

database_mmv= db.Database('database_mmv',df_mmv)


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.loc[df_pedestrian['age_opposite_mean'] != 999].copy()

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


## Variables and Betas

In [ ]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)


## Model for car crashes

### Baseline (constants only)

We first estimate the constants-only model. Its log-likelihood is the
denominator used for the rho-square statistics reported below and for the
likelihood-ratio test (see the LR-test section).


In [ ]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_carcrashes'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_carcrashes, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = model_cst_car.estimate()



### Full mixed-logit specification


In [ ]:
v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I +beta_age_I*age
      + beta_user_category_passenger_I * user_category_passenger
      + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I_bike * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
      + beta_point_of_impact_back_I_epmd* point_of_impact_back * vehicle_e_pmd
      + beta_vehicle_type_2_light_motorized_vehicle_I* vehicle_type_2_light_motorized_vehicle
      + beta_maneuver_2_overtaking_I * maneuver_2_overtaking
      + beta_maneuver_2_without_change_of_direction_I_bike * maneuver_without_change_of_direction * vehicle_bike
      + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
      + beta_intersection_no_intersection_I * (intersection_no_intersection==0)
      
)

v_fatality = (  constant_F
        + beta_user_category_passenger_I * user_category_passenger
        + beta_age_F * age
        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * vehicle_type_2_large_motorized_vehicle
        + beta_vehicle_type_2_light_motorized_vehicle_F* vehicle_type_2_light_motorized_vehicle
        + beta_lighting_conditions_night_with_street_lightings_on_F * lighting_conditions_night_with_street_lightings_on
        + beta_maneuver_2_turning_right_F * maneuver_2_turning_right
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
        + beta_lighting_conditions_night_without_street_lightings_F* lighting_conditions_night_without_street_lightings
        + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [ ]:
# Random-parameters model
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] 
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


# We integrate over B_TIME_RND using Monte-Carlo
logprob = log((prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_carcrashes,logprob)
model_car.modelName = "logit_car_crashes"

# Estimate the parameters. 
results_ml_carcrashes = model_car.estimate()


In [ ]:
results_ml_carcrashes.get_estimated_parameters()

## MMV

### Baseline (constants only)


In [ ]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [ ]:
model_name = 'InitialModel_mmv'



logprob_2 = models.loglogit(U, availability1, severity)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = model_cst_mmv.estimate()



### Full logit specification


In [ ]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female + gender_3_male)
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + beta_point_of_impact_back_I_epmd     * point_of_impact_back * vehicle_e_pmd
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
     + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_opposite_mean
    + beta_vehicle_2_e_pmd_I           * (vehicle_2_e_pmd + vehicle_3_e_pmd)
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [ ]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability1, severity)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv= model_mmv.estimate()
results_logit_mmv.get_estimated_parameters()

## Pedestrian

### Baseline (constants only)

Ordered-probit baseline with a flat utility (`continuous_value=0`) and the
single threshold `tau_1`.


In [ ]:
model_name = 'ordered_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = model_cst_pedes.estimate()


### Full ordered-probit specification


In [ ]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * (intersection_no_intersection==0)
    + beta_age_opposite_mean                             * age_opposite_mean
    + beta_gender_2_female                   * gender_2_female
    + beta_user_category_pedestrian          * user_category_pedestrian
    + beta_maneuver_2_turning_left           * user_category_pedestrian
                                            * (maneuver_2_turning_left + maneuver_2_turning_right)
    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [ ]:
model_name = 'ordered_probit_pedestrian'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_pedes.modelName = model_name
results_pedes = model_pedes.estimate()
results_pedes.get_estimated_parameters()

## Single-vehicle

### Baseline (constants only)

Ordered-probit baseline (`continuous_value=0`).


In [ ]:
model_name = 'ordered_logit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = model_cst_solo.estimate()
results_cst_solo.get_estimated_parameters()

### Full ordered-probit specification


In [ ]:
beta_user_category_passenger_mixed=beta_user_category_passenger_mean + beta_user_category_passenger_sd *X1


In [ ]:
utility_sv = (
   beta_age * age +
    beta_user_category_passenger * user_category_passenger +
beta_long_profile_slope *long_profile_slope +
beta_helmet_yes_ebike*helmet_yes*(vehicle_e_bike)
+ beta_number_of_passengers*user_category_driver*(number_of_passengers) 
)


In [ ]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob)
model_solo_2.modelName = model_name
results_solo_2 = model_solo_2.estimate()
results_solo_2.get_estimated_parameters()

## Likelihood-ratio test

The likelihood-ratio (LR) test compares two **nested** models -- a *restricted*
model (fewer parameters) against an *unrestricted* model that contains it as a
special case. Under the null hypothesis that the additional parameters are all
zero, the statistic

$$LR = -2\,(\,\ell_R - \ell_U\,)$$

is asymptotically chi-square distributed with degrees of freedom equal to the
number of additional parameters in the unrestricted model. A small p-value
therefore means the extra parameters jointly improve the fit.

Below, each estimated model is compared with its constants-only baseline. A
rejection of $H_0$ (small p-value) confirms that the explanatory variables
added on top of the constants explain the severity outcome significantly.


In [ ]:
def get_results(file_path):
    """Load a Biogeme `bioResults` object back from its pickle file."""
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return res.bioResults(data)


# Each pair: (full model results, constants-only results, label).
lr_pairs = [
    (results_ml_carcrashes, results_constant_car,    'Car crashes (mixed logit)'),
    (results_logit_mmv,     results_constant_mmv,    'MMV (logit)'),
    (results_pedes,         results_pedes_cst,       'Pedestrian (ordered probit)'),
    (results_solo_2,        results_cst_solo,        'Single-vehicle (ordered probit)'),
]

for full, restricted, label in lr_pairs:
    print(f'=== {label} ===')
    # Biogeme's helper: tests restricted (current object) vs. unrestricted.
    print(restricted.likelihood_ratio_test(full, 0.05))
    print()


## Out-of sample validation of the models

**Runtime warning.** Out-of-sample validation re-estimates each model on
every slice of the data


In [ ]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_carcrashes)
validationData_mmv= create_validation_data(df_mmv)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [ ]:
# Validate the model with the validation data for car
validation_results_car = model_car.validate(results_ml_carcrashes, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_carcrashes, validationData_car)


# Initialize variables to store log-likelihoods

loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (car)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_car += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (cars)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_car += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (cars)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





In [ ]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





In [ ]:

# Validate the model with the validation data for mmv
validation_results_pedes = model_pedes.validate(results_pedes, validationData_pedestrian)
validation_results_pedes_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_pedes= 0
loglike_constant_pedes = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_pedes):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_pedes += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_pedes_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_pedes += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on pedes (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_pedes)):
    validation_loglike = validation_results_pedes[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_pedes_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on pedes (slide {i+1}): {rho_square}')




In [ ]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')
